# Hermes × Colab worker (manager edition)
1. Run the auth cell and approve Google Drive access yourself.
2. Run the setup cell: it detects CPU/GPU/CUDA on this runtime.
3. Run the worker cell: it processes Hermes jobs queued in the
   Drive Vault (ColabJobs), writing heartbeats with device info,
   checkpoints, results and a progress log back to Drive.

**Not an always-on service.** Free Colab runtimes are disposable
and cannot be started from outside; Hermes keeps the queue on
Drive and resumes interrupted jobs when a runtime returns.
Never paste secrets, codes or cookies into this notebook or chat.

In [ ]:
# Step 1 -- run this cell FIRST and complete Google's official OAuth prompt
# yourself in the Colab/browser UI (sign in with the Google account that
# can access the Vault). Never share verification codes, passwords or
# cookies with anyone; authentication happens only in this UI.
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
print("Google Drive mounted at /content/drive")


In [ ]:
# Step 2: environment + device detection (CPU / GPU / CUDA).
# The runtime is disposable: everything durable is written to Drive.
import json
import os
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

VAULT_NAME = "ChatGPT-Obsidian-Vault"
VAULT = Path("/content/drive/MyDrive") / VAULT_NAME
QUEUE = VAULT / "06_Automation" / "Hermes" / "ColabJobs"

def detect_device():
    device = {"python": platform.python_version(),
              "platform": platform.platform(),
              "cpu_count": os.cpu_count(),
              "gpu": {"available": False, "name": None, "cuda": False}}
    try:
        import torch
        device["torch"] = torch.__version__
        if torch.cuda.is_available():
            device["gpu"] = {"available": True,
                             "name": torch.cuda.get_device_name(0),
                             "cuda": True,
                             "cuda_version": torch.version.cuda}
            # Real CUDA smoke computation.
            x = torch.ones(1024, device="cuda")
            device["gpu"]["cuda_smoke_sum"] = float((x @ x).item())
    except Exception as exc:
        device["torch_error"] = type(exc).__name__
    if not device["gpu"]["available"] and shutil.which("nvidia-smi"):
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=name",
                                  "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=20)
            name = out.stdout.strip().splitlines()[0] if out.stdout.strip() else None
            if name:
                device["gpu"] = {"available": True, "name": name, "cuda": False,
                                 "note": "reported by nvidia-smi; torch unavailable"}
        except Exception as exc:
            device["gpu_error"] = type(exc).__name__
    return device

DEVICE = detect_device()
QUEUE.mkdir(parents=True, exist_ok=True)
print(json.dumps(DEVICE, indent=2, ensure_ascii=False))


In [ ]:
# Step 3: Hermes Colab queue worker (user-started, bounded session).
# Executes allowlisted jobs only; writes heartbeat WITH device info,
# per-job checkpoints and a progress log to Drive. Stop the cell (or let
# the bounded session end) to stop the worker: queued jobs stay in Drive
# and Hermes requeues interrupted ones after backoff. This notebook does
# NOT keep a free runtime alive and cannot be started from outside.
import hashlib
import json
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

WORKER_MINUTES = 60
POLL_SECONDS = 15
MAX_JOBS = 100
INPUT_PREFIX = "06_Automation/Hermes/"
WORKER_ID = "colab-" + uuid.uuid4().hex[:12]
JOB_ID_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,127}$")
SEGMENT = re.compile(r"^[A-Za-z0-9_. -]+$")

def timestamp():
    return datetime.now(timezone.utc).isoformat()

def persist(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                    encoding="utf-8")

def log_event(event):
    line = json.dumps({"at": timestamp(), "worker_id": WORKER_ID, **event},
                      ensure_ascii=False)
    with open(QUEUE / "colab-worker-log.jsonl", "a", encoding="utf-8") as fh:
        fh.write(line + "\n")
    print(line, flush=True)

def health(state):
    persist(QUEUE / "worker.json", {
        "worker_id": WORKER_ID, "state": state, "last_seen_at": timestamp(),
        "device": DEVICE})

def valid_input(path):
    if not isinstance(path, str) or len(path) > 500 or not path.startswith(INPUT_PREFIX):
        raise ValueError("not a Hermes Vault input")
    parts = path.split("/")
    if any(s in ("", ".", "..") or not SEGMENT.fullmatch(s) for s in parts):
        raise ValueError("invalid path")
    if "\\" in path:
        raise ValueError("invalid path")
    result = VAULT.joinpath(*parts).resolve()
    if not result.is_relative_to((VAULT / "06_Automation" / "Hermes").resolve()):
        raise ValueError("input escapes allowed Hermes subtree")
    return result

def execute(spec):
    if not isinstance(spec, dict):
        raise ValueError("invalid job definition")
    kind = spec.get("type")
    if kind == "echo":
        message = spec.get("message", "")
        if not isinstance(message, str) or len(message) > 2000:
            raise ValueError("invalid echo message")
        return {"type": "echo", "message": message}
    if kind == "device_info":
        return {"type": "device_info", "device": DEVICE}
    if kind in ("text_stats", "hash_file"):
        name = spec.get("input")
        blob = valid_input(name).read_bytes()
        if len(blob) > 50 * 1024 * 1024:
            raise ValueError("file exceeds Colab worker maximum")
        result = {"type": kind, "input": name, "size": len(blob),
                  "sha256": hashlib.sha256(blob).hexdigest()}
        if kind == "text_stats":
            text = blob.decode("utf-8")
            result.update({
                "lines": text.count("\n") + (1 if text and not text.endswith("\n") else 0),
                "words": len(text.split()), "chars": len(text)})
        return result
    raise ValueError("unsupported job type")

CURRENT = {"job_id": None}

def process_one(folder):
    job_id = folder.name
    if not JOB_ID_PATTERN.fullmatch(job_id):
        return False
    status_file = folder / "status.json"
    job_file = folder / "job.json"
    if not job_file.is_file() or not status_file.is_file():
        return False
    try:
        status = json.loads(status_file.read_text(encoding="utf-8"))
        if status.get("state") != "queued" or status.get("job_id") != job_id:
            return False
        job = json.loads(job_file.read_text(encoding="utf-8"))
        if job.get("job_id") != job_id or job.get("schema") != 1:
            raise ValueError("job definition mismatch")
        # Resident jobs belong to the supervised Linux worker, never Colab.
        if (job.get("spec") or {}).get("executor", "colab") != "colab":
            return False
    except (OSError, json.JSONDecodeError, KeyError, ValueError):
        return False
    CURRENT["job_id"] = job_id
    persist(status_file, {"job_id": job_id, "state": "running",
                          "updated_at": timestamp(),
                          "attempts": status.get("attempts", 0)})
    persist(folder / "checkpoint.json",
            {"job_id": job_id, "stage": "executing", "updated_at": timestamp(),
             "worker_id": WORKER_ID})
    log_event({"event": "job_started", "job_id": job_id})
    try:
        payload = execute(job["spec"])
        result = {"job_id": job_id, "ok": True, "result": payload,
                  "worker_id": WORKER_ID, "device": DEVICE,
                  "finished_at": timestamp()}
        state = "done"
    except Exception as exc:
        result = {"job_id": job_id, "ok": False, "error": type(exc).__name__,
                  "worker_id": WORKER_ID, "finished_at": timestamp()}
        state = "failed"
    persist(folder / "result.json", result)
    persist(folder / "checkpoint.json",
            {"job_id": job_id, "stage": state, "updated_at": timestamp()})
    persist(status_file, {"job_id": job_id, "state": state,
                          "updated_at": timestamp(),
                          "attempts": status.get("attempts", 0)})
    log_event({"event": "job_finished", "job_id": job_id, "state": state,
               "ok": result["ok"]})
    CURRENT["job_id"] = None
    return True

completed = 0
deadline = time.monotonic() + min(max(WORKER_MINUTES, 1), 120) * 60
health("running")
log_event({"event": "worker_started", "device": DEVICE})
try:
    while time.monotonic() < deadline and completed < MAX_JOBS:
        for entry in sorted(QUEUE.iterdir()):
            if entry.is_dir() and process_one(entry):
                completed += 1
                if completed >= MAX_JOBS:
                    break
        health("running")
        if completed < MAX_JOBS:
            time.sleep(max(POLL_SECONDS, 5))
except KeyboardInterrupt:
    # Safe evacuation: an in-flight job is marked interrupted so Hermes
    # requeues it after backoff instead of losing it.
    if CURRENT["job_id"]:
        folder = QUEUE / CURRENT["job_id"]
        persist(folder / "checkpoint.json",
                {"job_id": CURRENT["job_id"], "stage": "interrupted",
                 "updated_at": timestamp(), "worker_id": WORKER_ID})
        persist(folder / "status.json",
                {"job_id": CURRENT["job_id"], "state": "interrupted",
                 "updated_at": timestamp()})
        log_event({"event": "job_interrupted", "job_id": CURRENT["job_id"]})
    print("Stopped by user. Queued jobs remain in Drive.")
finally:
    health("stopped")
    log_event({"event": "worker_stopped", "completed": completed})
print("Worker stopped. Completed jobs:", completed)
